In [84]:
import requests
import datetime
from pathlib import Path
import json
from bs4 import BeautifulSoup
import textwrap

URL = 'https://leetcode.com/graphql'
HEADERS = {
    'Content-Type':'application/json',
    'Referer':'https://leetcode.com'
}


QUERY_GETDAILYPROBLEMNUMBER = '''
    query getDailyProblem {
        activeDailyCodingChallengeQuestion {
            question {
                questionFrontendId
            }
        }
    }
    '''

    
QUERY_GETDAILYPROBLEM = '''
query getDailyProblem {
    activeDailyCodingChallengeQuestion {
        date
        link
        question {
            questionFrontendId
            title
            titleSlug
            difficulty
            content

            exampleTestcaseList
            
            codeSnippets {
                lang
                langSlug
                code
            }
        }

    }
}
'''

QUERY_GETTITLESLUG = '''
query getProblemSlug($search: String) {
  problemsetQuestionList: questionList(
    categorySlug: ""
    limit: 1
    skip: 0
    filters: { searchKeywords: $search }
  ) {
    questions: data {
      frontendQuestionId: questionFrontendId
      link
      title
      titleSlug
      difficulty
      content

      exampleTestcaseList

      codeSnippets {
        lang
        langSlug
        code
      }
    }
  }
}
'''

In [83]:
def send_leetcode_request(query:str|dict) -> requests.Response:
    if isinstance(query, str):
        payload = {'query':query}
    elif isinstance(query, dict):
        payload = query
    else:
        raise ValueError(f'Expected `query` as type str or dict, got `{type(query)}`')


    resp = requests.post(
        url=URL,
        headers=HEADERS,
        json=payload,
    )
    # resp.raise_for_status()

    return resp



def get_problem_number() -> int:
    resp = send_leetcode_request(QUERY_GETDAILYPROBLEMNUMBER)
    qid = (
        resp.json()
            .get('data',{})
            .get('activeDailyCodingChallengeQuestion',{})
            .get('question',{})
            .get('questionFrontendId','-1')
    )
    qid = int(qid)
    if qid == -1: raise ValueError('`get_problem_number` - issue parsing data')

    return qid

def check_exists(qid:int) -> bool:
    loc = Path.cwd()
    test_loc = Path.cwd() / 'Test_Cases'
    py_match = list( loc.glob(f'p{qid:04d}*.py') )
    test_match = list( test_loc.glob(f'p{qid:04d}*.json') )

    flag = False
    if py_match: 
        print('Python file found')
        flag = True
    if test_match:
        print('Test cases file found')
        flag = True
    
    return flag


def export_python_snippet(qdata:dict, link:str) -> None:
    code_snippets = qdata.get('codeSnippets')

    python_snippet = ''
    for cs in code_snippets:
        if cs.get('langSlug') == 'python3':
            python_snippet = cs.get('code')
        elif cs.get('langSlug') == 'python' and python_snippet != '':
            python_snippet = cs.get('code')

    problem_number = int(qdata.get('questionFrontendId'))
    problem_title  = qdata.get('title')
    problem_diff   = qdata.get('difficulty')

    python_code_sugar = f'''
{problem_number:04d}. {problem_title} [{problem_diff}]
{link} 


{python_snippet}
    '''.strip()


    python_file = Path.cwd() / f'{problem_number:04d}.py'
    assert not python_file.exists(), 'Python file exists, delete to overwrite'
    python_file.write_text(python_code_sugar)


def export_test_cases(qdata:dict) -> None:

    problem_number = int(qdata.get('questionFrontendId'))
    test_file = Path.cwd() / f'Test_Cases/p{problem_number:04d}.json'
    assert not test_file.exists(), 'Test file exists, delete to overwrite'

    test_cases = list()
    for ex in qdata.get('exampleTestcaseList'):
        case = [
            eval(ex),
            None # outputs to be filled in later
        ]
        test_cases.append(case)

    test_file.write_text(json.dumps(test_cases, indent=4))


In [3]:
def get_todays_problem():
    resp = send_leetcode_request(QUERY_GETDAILYPROBLEM)
    data = resp.json().get('data',{}).get('activeDailyCodingChallengeQuestion')
    assert data is not None, '`get_todays_problem` - No results found'

    link = f'https://leetcode.com' + data.get('link')
    qdata = data.get('question')
    assert qdata is not None, '`get_todays_problem` - No question data'
    
    
    return link, qdata

In [85]:
# def get_problem_by_number():
payload = {
    'query': QUERY_GETTITLESLUG,
    'variables': {'search':'2747'}
}

resp = send_leetcode_request(payload)

In [87]:
resp.text

'{"errors":[{"message":"Cannot query field \\"link\\" on type \\"QuestionNode\\".","locations":[{"line":11,"column":7}]}]}'

In [19]:
data = (
    resp.json()
        .get('data',{})
        .get('problemsetQuestionList',{})
        .get('questions',[])
)
if len(data) == 0:
    # Error
    print('No results found')
    assert False
elif len(data) > 1:
    # Error
    print('Multiple results found')
    assert False

question_data = data[0]


In [24]:
qid = int(question_data.get('frontendQuestionId'))

title = question_data.get('title')
difficulty = question_data.get('difficulty')
content = question_data.get('content')

test_cases = question_data.get('exampleTestcaseList')


# question_data

In [88]:
data

[{'frontendQuestionId': '2747',
  'title': 'Count Zero Request Servers',
  'titleSlug': 'count-zero-request-servers',
  'difficulty': 'Medium',
  'content': '<p>You are given an integer <code>n</code> denoting the total number of servers and a <strong>2D</strong> <strong>0-indexed </strong>integer array <code>logs</code>, where <code>logs[i] = [server_id, time]</code> denotes that the server with id <code>server_id</code> received a request at time <code>time</code>.</p>\n\n<p>You are also given an integer <code>x</code> and a <strong>0-indexed</strong> integer array <code>queries</code>.</p>\n\n<p>Return <em>a <strong>0-indexed</strong> integer array</em> <code>arr</code> <em>of length</em> <code>queries.length</code> <em>where</em> <code>arr[i]</code> <em>represents the number of servers that <strong>did not receive</strong> any requests during the time interval</em> <code>[queries[i] - x, queries[i]]</code>.</p>\n\n<p>Note that the time intervals are inclusive.</p>\n\n<p>&nbsp;</p>\

In [29]:
eval(test_cases[0].replace('\n',','))

(3, [[1, 3], [2, 6], [1, 5]], 5, [10, 11])

In [ ]:
soup = BeautifulSoup(content, 'html.parser')
for sup in soup.find_all('sup'):
    repl = f'^{sup.text}'
    sup.replace_with(repl)

content_str = soup.text
content_str_wrap = [
    '\n'.join(textwrap.wrap(line, width=80)) 
    for line in content_str.split('\n')
]
print('\n'.join(content_str_wrap))



You are given an integer n denoting the total number of servers and a 2D
0-indexed integer array logs, where logs[i] = [server_id, time] denotes that the
server with id server_id received a request at time time.
You are also given an integer x and a 0-indexed integer array queries.
Return a 0-indexed integer array arr of length queries.length where arr[i]
represents the number of servers that did not receive any requests during the
time interval [queries[i] - x, queries[i]].
Note that the time intervals are inclusive.

Example 1:

Input: n = 3, logs = [[1,3],[2,6],[1,5]], x = 5, queries = [10,11]
Output: [1,2]
Explanation:
For queries[0]: The servers with ids 1 and 2 get requests in the duration of [5,
10]. Hence, only server 3 gets zero requests.
For queries[1]: Only the server with id 2 gets a request in duration of [6,11].
Hence, the servers with ids 1 and 3 are the only servers that do not receive any
requests during that time period.


Example 2:

Input: n = 3, logs = [[2,4],[2,1]

In [64]:
python_snippet = question_data.get('codeSnippets')[2].get('code')

In [65]:
print(python_snippet)

class Solution:
    def countServers(self, n: int, logs: List[List[int]], x: int, queries: List[int]) -> List[int]:
        


In [67]:
import re

In [82]:
res = re.search(pattern=r'def (.*)\(', string=python_snippet)


res.group(1)

'countServers'